# EditFlowSR demo

This notebook recovers closed-form expressions from observed data using EditFlowSR, which iteratively edits a random token sequence into an expression through insertion, deletion, and substitution.

## 1. Install

From the repository root:

```bash
pip install -e .
```

In [ ]:
import sys

import numpy as np
import torch

import editflowsr
from editflowsr.inference import (
    decode_prediction,
    evaluate_tree_candidate,
    load_env_from_checkpoint,
    load_models_from_checkpoint,
    refine_tree_with_bfgs,
    sample_with_euler,
    score_candidate,
)

print("python:", sys.version)
print("editflowsr loaded from:", editflowsr.__file__)

## 2. Checkpoint

Download the pretrained checkpoint (see ) and set the path below.

In [ ]:
from pathlib import Path

WEIGHTS_PATH = "weights/editflowsr_pretrained.pth"

assert Path(WEIGHTS_PATH).is_file(), (
    f"checkpoint not found: {WEIGHTS_PATH}. "
    "Download it with download_weights.py first."
)
print("checkpoint:", WEIGHTS_PATH)

## 3. Toy cases

Two one-dimensional functions sampled on 100 points. The model only sees the points, never the generating expressions.

In [ ]:
x = np.linspace(-3.0, 3.0, 100).astype(np.float32).reshape(-1, 1)

cases = [
    {"name": "x**2 + 2*cos(x)", "y": (x ** 2 + 2.0 * np.cos(x)).reshape(-1, 1)},
    {"name": "sin(x) * exp(x/2)", "y": (np.sin(x) * np.exp(x / 2.0)).reshape(-1, 1)},
]

for case in cases:
    print(case["name"], "| y range:", case["y"].min().round(3), "..", case["y"].max().round(3))

In [ ]:
env, params, checkpoint_data = load_env_from_checkpoint(WEIGHTS_PATH)
embedder, encoder, decoder = load_models_from_checkpoint(checkpoint_data, env)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for model in (embedder, encoder, decoder):
    model.to(device).eval()
print("device:", device)

## 4. Recover expressions

For each case, the sampler draws a beam of candidate token sequences, decodes each candidate into an expression, refines its constants with BFGS, and keeps the candidate with the best R².

In [ ]:
BEAM_SIZE = 10
N_STEPS = 100

for case in cases:
    y = case["y"].astype(np.float32)
    with torch.no_grad():
        result_tokens, _ = sample_with_euler(
            embedder, encoder, decoder, x, y, env, params,
            scheduler_a=2.0, scheduler_b=0.5,
            n_steps=N_STEPS, sample_idx=0,
            device=device, beam_size=BEAM_SIZE,
        )
    candidates = []
    for beam_idx in range(result_tokens.shape[0]):
        decoded = decode_prediction(result_tokens[beam_idx], env)
        tree = decoded["predicted_tree"]
        if tree is None:
            continue
        candidate = evaluate_tree_candidate(
            name=f"beam{beam_idx}", tree=tree,
            x_to_fit=x, y_to_fit=y, env=env,
            metrics="r2,r2_zero,_complexity",
            true_tree=None, error=decoded["error"],
        )
        candidates.append(candidate)
        refined_tree, refine_error = refine_tree_with_bfgs(tree, x, y, env)
        if refined_tree is not None:
            candidates.append(evaluate_tree_candidate(
                name=f"beam{beam_idx}+bfgs", tree=refined_tree,
                x_to_fit=x, y_to_fit=y, env=env,
                metrics="r2,r2_zero,_complexity", true_tree=None,
                error=refine_error,
            ))
    valid = [c for c in candidates if c["tree"] is not None and c["metrics"].get("r2") is not None]
    best = max(valid, key=lambda c: score_candidate(c["metrics"])) if valid else None
    print(f"
{case['name']}")
    if best is not None:
        print("  recovered:", best["tree"].infix())
        print("  R2:", round(best["metrics"]["r2"], 4), "| complexity:", best["metrics"].get("_complexity"))
    else:
        print("  no valid candidate")